In [3]:
import pandas as pd
import json

In [13]:
SAVE_DIR = "./../data/datasets/"

In [18]:
def unify_dataset_schema(file_json, file_csv, dataset):
    # create df from json
    df1 = pd.read_json(file_json)
    df1 = df1.transpose()
    df2 = pd.read_csv(file_csv)
    # Initialize the new columns
    df1["Indication_approved_extracted"] = None
    df1["Indication_requested_extracted"] = None
    df1["Marketing_authorisation_holder_extracted"] = None
    
    for row in df1.iterrows():
        document_name = row[1].get("Document_name")
        if document_name in df2["Document_name"].values:
            matching_rows = df2[df2["Document_name"] == document_name]
            if not matching_rows.empty:
                matching_row = matching_rows.iloc[0]
                
                df1.loc[row[0], "Indication_approved_extracted"] = matching_row.get("Indication_approved_extracted", None)
                df1.loc[row[0], "Indication_requested_extracted"] = matching_row.get("Indication_requested_extracted", None)
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = matching_row.get("Marketing_authorisation_holder_extracted", None)
                df1.loc[row[0], "Dataset"] = dataset
            else:
                # No matching rows found
                df1.loc[row[0], "Indication_approved_extracted"] = None
                df1.loc[row[0], "Indication_requested_extracted"] = None
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None
        else:
            # Document_name not in df2
            df1.loc[row[0], "Indication_approved_extracted"] = None
            df1.loc[row[0], "Indication_requested_extracted"] = None
            df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None

    df1 = df1.reindex(sorted(df1.columns), axis=1)

    return df1

# EMA

In [20]:
filepath_json = "./../inference/combined/EMA_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/EMA_manually_cleaned.csv"
merged_EMA = unify_dataset_schema(filepath_json, filepath_csv, dataset="EMA")
merged_EMA.to_csv(SAVE_DIR + "EMA.csv", index=False)
with open(SAVE_DIR + "EMA.json", "w", encoding="utf-8") as out:
    json.dump(merged_EMA.to_dict(orient="index"), out, indent=4, sort_keys=True)

# Swissmedic

In [21]:
filepath_json = "./../inference/combined/SWISSMEDIC_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/SWISSMEDIC_manually_cleaned.csv"
merged_SWISSMEDIC = unify_dataset_schema(filepath_json, filepath_csv, dataset="SWISSMEDIC")
merged_SWISSMEDIC.to_csv(SAVE_DIR + "SWISSMEDIC.csv", index=False)
with open(SAVE_DIR + "SWISSMEDIC.json", "w", encoding="utf-8") as out:
    json.dump(merged_SWISSMEDIC.to_dict(orient="index"), out, indent=4, sort_keys=True)

# Japan

In [22]:
filepath_json = "./../inference/combined/JAPAN_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/JAPAN_manually_cleaned.csv"
merged_JAPAN = unify_dataset_schema(filepath_json, filepath_csv, dataset="JAPAN")
merged_JAPAN.to_csv(SAVE_DIR + "JAPAN.csv", index=False)
with open(SAVE_DIR + "JAPAN.json", "w", encoding="utf-8") as out:
    json.dump(merged_JAPAN.to_dict(orient="index"), out, indent=4, sort_keys=True)

# Australia

In [23]:
filepath_json = "./../inference/combined/AUSTRALIA_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/AUSTRALIA_manually_cleaned.csv"
merged_AUSTRALIA = unify_dataset_schema(filepath_json, filepath_csv, dataset="AUSTRALIA")
merged_AUSTRALIA.to_csv(SAVE_DIR + "AUSTRALIA.csv", index=False)
with open(SAVE_DIR + "AUSTRALIA.json", "w", encoding="utf-8") as out:
    json.dump(merged_AUSTRALIA.to_dict(orient="index"), out, indent=4, sort_keys=True)

# FDA

In [24]:
with open("./../inference/combined/FDA_manually_cleaned.json", "r") as f1:
    data1 = json.load(f1)
with open("./../data/FDA/with_extracted_data_drug_class/FDA.json", "r") as f2:
    data2 = json.load(f2)

lookup = {}
for entry in data2.values():
    ma_number = entry.get("MA_Number")
    drug_class = entry.get("Non_proprietary_name_extracted")
    if ma_number:
        lookup[ma_number] = drug_class

# Enrich data1
for key, entry in data1.items():
    ma_number = entry.get("Marketing_authorisation_number")
    if ma_number and ma_number in lookup:
        entry["Drug_class"] = lookup[ma_number]
        entry["Application_date"] = None
        entry["Application_year"] = None
        entry["Document_name"] = None
        entry["Indication_approved"] = entry.get("Indications_and_usage")
        entry.pop("Indications_and_usage", None) 
        entry["Indication_approved_extracted"] = None
        entry["Indication_requested"] = None
        entry["Indication_requested_extracted"] = None
        entry["Procedure_number"] = None
        entry["Referral_body"] = entry.get("Referral")
        entry.pop("Referral", None)
        entry["Dataset"] = "FDA"
        entry.pop("Origin", None)
    if not ma_number:
        print(f"MA_Number not found for entry: {entry}")
 
# Save to new JSON file
output_path = "./../data/datasets/FDA.json"
with open(output_path, "w", encoding="utf-8") as out:
    json.dump(data1, out, indent=4, sort_keys=True)

print(f"Enriched file saved to {output_path}")

# Save as CSV just in case
df = pd.DataFrame(data1).transpose()
df.to_csv("./../data/datasets/FDA.csv", encoding="utf-8")



Enriched file saved to ./../data/datasets/FDA.json


In [25]:
df.columns

Index(['Marketing_authorisation_number', 'Drug_name', 'Non_proprietary_name',
       'Marketing_authorisation_holder', 'Pharmaceutical_form',
       'Administration_route', 'Decision', 'Decision_date', 'Decision_year',
       'Current_status', 'Nonclinical_abridged', 'Orphan_drug_status',
       'Marketing_authorisation_holder_extracted', 'Drug_class',
       'Disease_class(es)', 'Disease_name(s)', 'Application_date',
       'Application_year', 'Document_name', 'Indication_approved',
       'Indication_approved_extracted', 'Indication_requested',
       'Indication_requested_extracted', 'Procedure_number', 'Referral_body',
       'Dataset'],
      dtype='object')

# HealthCanada

In [12]:
# waiting for data extraction with LLM

In [ ]:
# Example: How to order JSON fields alphabetically

# Method 1: Using sort_keys=True in json.dump()
sample_data = {"z_field": "value1", "a_field": "value2", "m_field": "value3"}
print("Original order:", list(sample_data.keys()))

# Save with sorted keys
with open("./test_sorted.json", "w") as f:
    json.dump(sample_data, f, indent=4, sort_keys=True)

# Read back to see sorted order
with open("./test_sorted.json", "r") as f:
    content = f.read()
    print("Sorted JSON content:")
    print(content)

# Method 2: For pandas DataFrame - sort columns alphabetically
if 'merged_EMA' in locals():
    print("\nEMA columns in alphabetical order:")
    sorted_df = merged_EMA.reindex(sorted(merged_EMA.columns), axis=1)
    print(sorted_df.columns.tolist())

In [ ]:
# Function to save DataFrame as alphabetically sorted JSON
def save_df_as_sorted_json(df, filepath):
    """Save DataFrame as JSON with alphabetically sorted fields"""
    # Sort columns alphabetically
    df_sorted = df.reindex(sorted(df.columns), axis=1)
    
    # Convert to dict (transpose to get original JSON structure)
    data_dict = df_sorted.T.to_dict()
    
    # Create directory if needed
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    
    # Save with sorted keys
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data_dict, f, indent=4, sort_keys=True)
    
    print(f"Saved {filepath} with {len(data_dict)} records and sorted fields")
    return df_sorted

# Example usage - save EMA with sorted fields
if 'merged_EMA' in locals():
    sorted_ema = save_df_as_sorted_json(merged_EMA, "./data/datasets/EMA_sorted.json")